# 07 · Legacy regression comparison

Diffs the tcren-reproduced outputs in `results_new/` against the 2022 mir/R baselines in
`data_legacy/` (the regression/debugging surface). Each row is one paired file with the
row-set match and the maximum numeric difference. Run the analysis notebooks first so that
`results_new/` is populated.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, polars as pl
from tcren.paper import compare
from tcren.paper.helpers import _read_any

CONTACT_KEYS = ['pdb.id','chain.type.from','region.type.from','residue.index.from',
                'residue.index.to','residue.aa.from','residue.aa.to']

In [ ]:
# Contacts: tcren vs mir, restricted to the structures present in both (exact set match)
new = _read_any('results_new/contacts_native.csv')
old = _read_any('data_legacy/contact_maps_PDB.csv.gz')
shared = sorted(set(new['pdb.id'].to_list()) & set(old['pdb.id'].to_list()))
new_s = new.filter(pl.col('pdb.id').is_in(shared)).select(CONTACT_KEYS).unique()
old_s = old.filter(pl.col('pdb.id').is_in(shared)).select(CONTACT_KEYS).unique()
g = set(map(tuple, new_s.rows())); o = set(map(tuple, old_s.rows()))
rows = [{'artifact': 'contacts (vs mir)', 'n_structures': len(shared),
         'rows_new': len(g), 'rows_old': len(o), 'only_new': len(g-o), 'only_old': len(o-g),
         'status': 'pass' if g == o else 'FAIL'}]
rows[-1]

In [ ]:
# Derived potential: tcren (HF subset) vs published — correlation (not exact: different structure set)
new_pot = _read_any('results_new/TCRen_native.csv')
old_pot = _read_any('data_legacy/TCRen_potential.csv.gz').rename({'TCRen': 'value'})
j = new_pot.join(old_pot, on=['residue.aa.from','residue.aa.to'], suffix='_pub')
r = float(np.corrcoef(j['value'].to_numpy(), j['value_pub'].to_numpy())[0, 1])
rows.append({'artifact': 'TCRen potential (vs published)', 'n_structures': None,
             'rows_new': new_pot.height, 'rows_old': old_pot.height,
             'only_new': None, 'only_old': None,
             'status': f'r={r:.3f}' + (' pass' if r > 0.9 else ' CHECK')})
rows[-1]

In [ ]:
# Regression summary table
report = pl.DataFrame(rows)
report.write_csv('results_new/regression_report.csv')
report